# Résoudre `3_free` avec OR-Tools
Troisième jeu de données : 5 000 modèles et 20 000 datasets. On commence par regarder **ce qui change** par rapport à `2_medium`, parce que le modèle mathématique doit suivre les données.

**Version 3.** Ce notebook n'implémente plus rien lui-même : il appelle le package `ia_responsable` (dossier `src/`). Le récit et les expériences sont ceux de la version précédente ; le code est celui que teste `make test` et qu'utilise `make solve`. Les fonctions sont commentées dans les modules indiqués entre parenthèses.

**Durée d'exécution** : moins d'une minute.

## 1. Charger l'instance

In [1]:
from pathlib import Path
import json, time, random
from collections import Counter
import pandas as pd

from ia_responsable import load
from ia_responsable.cli import official_score
from ia_responsable.fill import fill, give, new_pools, no_full_models
from ia_responsable.model import solve
from ia_responsable.packing import REFERENCE_ORDERS, noisy_orders, pack
from ia_responsable.pipeline import plan_value, run
from ia_responsable.repair import State, complete, repair

pd.set_option("display.width", 150)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
inst = load(ROOT / "datasets" / "3_free.json")
WINNER = 523_758


def score(plan):
    """Score officiel (test_solution.py). Lève une erreur si le plan est invalide."""
    s, valid, message = official_score(ROOT, inst.text, plan)
    assert valid, message
    return s


def solver_row(info, sel):
    return dict(statut=info.status, objectif=info.objective, borne=round(info.bound, 1), écart_prouvé=round(info.gap, 1),
                secondes=round(info.seconds), pleins=len(sel.full), demi=len(sel.half), sources=len(sel.sources))


def failed_full(sel, plan):
    """Modèles choisis à pleine valeur qui n'ont pas eu leurs données libres."""
    data = {}
    for d, m in plan["dataMappings"]:
        data.setdefault(m, []).append(d)
    return [m for m in sel.full if m not in data or any(inst.copyrighted[d] for d in data[m])]


def save_if_better(plan):
    out = ROOT / "solutions" / f"{inst.name}_main.json"
    new = score(plan)
    current = official_score(ROOT, inst.text, json.loads(out.read_text()))[0] if out.exists() else 0
    if new > current:
        out.write_text(json.dumps(plan))
        print(f"{out.name} mis à jour : {current:,} -> {new:,}")
    else:
        print(f"{out.name} inchangé ({current:,}) ; ce notebook a obtenu {new:,}")


print(f"{inst.name} : {len(inst.value)} modèles, {len(inst.size)} datasets | gagnant du concours : {WINNER:,}")

3_free : 5000 modèles, 20000 datasets | gagnant du concours : 523,758


## 2. Ce qui change

In [2]:
share = inst.energy_cap / sum(inst.energy.values())
demand = {t: sum(low.get(t, 0) for low in inst.lower.values()) for t in "ntic"}
print(f"datasets sous droits : {sum(inst.copyrighted.values())} sur {len(inst.size)}")
print(f"énergie de tous les modèles : {1 / share:.2f} × le plafond | modèles impossibles : {len(inst.impossible)}")
pd.DataFrame({"offre": inst.total_supply, "demande_probable": {t: round(demand[t] * share) for t in "ntic"},
              "offre_sur_demande_probable": {t: round(inst.total_supply[t] / (demand[t] * share), 1) for t in "ntic"}})

datasets sous droits : 0 sur 20000
énergie de tous les modèles : 7.95 × le plafond | modèles impossibles : 94


,offre,demande_probable,offre_sur_demande_probable
n,24727061,7460151,3.3
t,24952470,7569777,3.3
i,25957507,7455078,3.5
c,24644719,7367177,3.3


**Lecture**
- **Aucun dataset n'est sous droits** : le malus de demi-valeur ne vient plus que des sources. La rareté des données libres, qui coûtait cher sur `2_medium`, **disparaît**.
- L'énergie de tous les modèles fait **8 fois le plafond**.
- Les modèles qu'on pourra entraîner demanderont environ un tiers de l'offre : **les données ne sont plus une contrainte.**

Le problème devient un **sac à dos 0-1** sur l'énergie, plus les sources pour les modèles impossibles. La programmation dynamique, méthode exacte classique du sac à dos, est impossible ici : elle demanderait un tableau indexé par l'énergie, soit $3 \times 10^{14}$ cases. Un solveur MIP s'appuie sur la relaxation linéaire et n'est pas gêné par la taille des coefficients.

## 3. Résoudre : attention au mot « optimal »
Le package règle la **tolérance d'arrêt** de SCIP à zéro (`model.py`). Pourquoi ? Dans la version précédente de ce notebook, on avait lancé SCIP avec ses réglages par défaut : il annonçait « OPTIMAL » à **523 747**, 11 points sous le gagnant. Par défaut, SCIP s'arrête dès que l'écart relatif passe sous **0,01 %**, et 0,01 % de 523 000 représente jusqu'à 52 points : de quoi perdre la première place.

**« Optimal » veut toujours dire « optimal à la tolérance près ».**

In [3]:
sel, info = solve(inst, seconds=120)
pd.DataFrame([solver_row(info, sel)], index=["SCIP, tolérance nulle"])

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
"SCIP, tolérance nulle",OPTIMAL,523758,523758.0,0.0,4,1322,9,9


**Lecture** : avec une tolérance nulle, SCIP prouve l'optimum en quelques secondes : la solution et la borne coïncident.

## 4. Attribuer les datasets
Premier essai : l'ordre « petits besoins », celui du notebook 11.

In [4]:
results = {}
plan_a, n_failed = pack(inst, sel, REFERENCE_ORDERS["petits besoins"])
lost = sorted(set(sel.full + sel.half) - {m for _, m in plan_a["dataMappings"]} - {m for _, m in plan_a["modelMappings"]})
print("modèles non servis :", [(m, inst.needs[m]) for m in lost])
results["A : petits besoins"] = score(plan_a)
results

modèles non servis : [(2107, [('i', 1, 1995), ('n', 1, 9)])]


{'A : petits besoins': 523663}

**Lecture** : un modèle n'est pas servi. Regardez ses besoins : il lui faut entre 1 et 9 unités de type `n`, et seuls trois datasets `n` sont assez petits (tailles 4, 5 et 8). D'autres modèles servis avant eux, dont l'intervalle acceptait aussi de petites tailles, les ont pris. Quand une **cible** de source n'est pas servie, toute sa grappe (cible et sources) est abandonnée : une cible mal servie invaliderait toute la soumission.

Le modèle agrégé ne voit pas cette **concurrence pour les petits datasets** : pour lui, il y a des millions d'unités disponibles. **L'agrégation oublie la granularité.**

Deuxième essai : **servir d'abord les plus contraints**, règle classique de l'ordonnancement.

In [5]:
plan_b, n_failed = pack(inst, sel, REFERENCE_ORDERS["plus contraints"])
results["B : plus contraints"] = score(plan_b)
results

{'A : petits besoins': 523663, 'B : plus contraints': 523758}

## 5. Et sans solveur ?
La règle gloutonne la plus simple du notebook 09 : la meilleure densité `valeur / énergie` d'abord, sans source.

In [6]:
def greedy_density():
    pools, energy, mappings = new_pools(inst), 0, []
    for m in sorted((m for m in inst.value if m not in inst.impossible), key=lambda m: -inst.value[m] / inst.energy[m]):
        if energy + inst.energy[m] <= inst.energy_cap:
            got = give(inst, pools, inst.needs[m], (0, 1))
            if got is not None:
                mappings += [[d, m] for d in got]
                energy += inst.energy[m]
    return {"dataMappings": mappings, "modelMappings": []}

results["glouton par densité (sans solveur)"] = score(greedy_density())
summary = pd.Series(results, name="score").to_frame()
summary.loc["gagnant du concours"] = WINNER
summary.loc["optimum prouvé du modèle"] = round(info.bound)
summary["écart_au_gagnant"] = summary.score - WINNER
summary.sort_values("score")

,score,écart_au_gagnant
glouton par densité (sans solveur),523484,-274
A : petits besoins,523663,-95
B : plus contraints,523758,0
gagnant du concours,523758,0
optimum prouvé du modèle,523758,0


**Lecture** : l'ordre « plus contraints » atteint **exactement le score du gagnant**, qui est aussi l'**optimum prouvé** du modèle. Le glouton par densité est déjà très bon, mais il lui manque des points que seul un arbitrage global trouve : le dernier morceau d'énergie, et les appariements source-cible.

In [7]:
save_if_better(plan_b)

3_free_main.json inchangé (523,758) ; ce notebook a obtenu 523,758


## À retenir
1. **Le modèle suit les données.** Toutes les données sont libres : le problème devient un sac à dos sur l'énergie.
2. **« Optimal » veut dire « optimal à la tolérance près ».**
3. **L'agrégation oublie la granularité** : ici, quelques petits datasets que tout le monde veut.
4. **Servir d'abord les plus contraints.** Sur `2_medium`, cet ordre n'est pas toujours le meilleur : le package essaie toujours les deux.
5. **Un solveur exact sur un problème bien modélisé est imbattable** : optimum prouvé en quelques secondes.